# SIGMOD Exp 2: Crossover Analysis

This notebook fixes a read-heavy maintained-state profile and sweeps three factors that matter for `SNAP` vs `IVMH` vs MVHT tradeoffs:

1. Delta-scan share
2. Update intensity
3. History-scan reuse share

The output is a 3-panel line figure over total latency.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pandas', 'matplotlib', 'numpy'])
print('done')

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
import importlib
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)
sys.path.append(str(ROOT / 'benches' / 'hash_join' / 'htap_simulation'))

from sigmod_exp_common import (
    TOL,
    SIGMOD_BUCKET_NUM,
    SIGMOD_HTAP_TXN_COUNT,
    SIGMOD_HTAP_WAREHOUSE_COUNT,
    SIGMOD_REPEAT,
    SIGMOD_TRIM,
    apply_paper_style,
    display_name,
    ensure_dirs,
    run_checked,
)
from bench_script_functions import parse_result

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp2_crossover').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

BIN = ROOT / 'target' / 'release' / 'htap_wkld'
TABLE_TYPES = ['naive', 'ivmh', 'heap', 'chain', 'par']
BASELINES = {'naive', 'ivmh'}
TX_MAP = {'MarkTs': 'BuildSnap', 'DelSc': 'DeltaScan'}

CONFIG = {
    'warehouse_count': SIGMOD_HTAP_WAREHOUSE_COUNT,
    'txn_count': SIGMOD_HTAP_TXN_COUNT,
    'bucket_num': SIGMOD_BUCKET_NUM,
    'update_ratio': 0.003,
    'probe_ratio': 0.0001,
    'txn_gc_ratio': 0.05,
    'repeat': SIGMOD_REPEAT,
    'trim': SIGMOD_TRIM,
    'force_rerun': False,
}

SWEEP = {
    'analytical_ratio': 0.80,
    'delta_values': [0.00, 0.05, 0.10, 0.15, 0.20, 0.25],
    'update_values': [0.01, 0.05, 0.10, 0.15, 0.20],
    'reuse_values': [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
}

REPEAT = CONFIG['repeat']
TXN_NUM = CONFIG['txn_count']
BASE_ARGS = [
    '--txn-count', str(CONFIG['txn_count']),
    '--warehouse-count', str(CONFIG['warehouse_count']),
    '--update-ratio', str(CONFIG['update_ratio']),
    '--probe-ratio', str(CONFIG['probe_ratio']),
    '--bucket-num', str(CONFIG['bucket_num']),
]
PLOT_SERIES = [
    ('naive', ''),
    ('ivmh', ''),
    ('heap', 'Write Repair'),
    ('chain', 'Write Repair'),
    ('par', 'Write Repair'),
]
STYLE = {
    ('naive', ''): ('SNAP', TOL['red'], ':', 'x'),
    ('ivmh', ''): ('IVMH', TOL['yellow'], '--', 'P'),
    ('heap', 'Write Repair'): ('MONO-WR', TOL['blue'], '-', 'o'),
    ('chain', 'Write Repair'): ('DUAL-WR', TOL['cyan'], '-', 's'),
    ('par', 'Write Repair'): ('EPOCH-WR', TOL['green'], '-', 'D'),
}


def token(value):
    if isinstance(value, float):
        return f'{value:g}'.replace('.', 'p')
    return str(value)


RUN_TAG = '_'.join([
    f"wc{token(CONFIG['warehouse_count'])}",
    f"tc{token(CONFIG['txn_count'])}",
    f"bn{token(CONFIG['bucket_num'])}",
    f"ur{token(CONFIG['update_ratio'])}",
    f"pr{token(CONFIG['probe_ratio'])}",
    f"gc{token(CONFIG['txn_gc_ratio'])}",
    f"rep{token(CONFIG['repeat'])}",
])

print('ROOT   :', ROOT)
print('BIN    :', BIN)
print('OUTDIR :', DATA_DIR)
print('CONFIG :', CONFIG)
print('TAG    :', RUN_TAG)


In [ ]:
print('Building htap_wkld...')
run_checked(['cargo', 'build', '--release', '--bin', 'htap_wkld'], ROOT)
print('Build OK')

In [ ]:
def merge_args(base, extra):
    merged = {}
    for args in (base, extra):
        it = iter(args)
        for token in it:
            merged[token] = next(it)
    out = []
    for k, v in merged.items():
        out.extend([k, v])
    return out


def trim_trial_runs(df):
    trim = CONFIG['trim']
    if trim <= 0 or df.empty or 'trial' not in df.columns:
        return df
    totals = (
        df.groupby('trial', as_index=False)['duration_ms']
        .sum()
        .sort_values('duration_ms')
    )
    if len(totals) <= 2 * trim:
        return df
    keep = set(totals.iloc[trim:len(totals) - trim]['trial'])
    return df[df['trial'].isin(keep)].copy()


def collapse_repairs(df, table_type):
    if table_type in BASELINES:
        collapsed = df.groupby(['table_type'], as_index=False)['duration_ms'].mean()
        collapsed['repair_type'] = ''
        return collapsed[['table_type', 'repair_type', 'duration_ms']]
    return df


def run_single(extra_args):
    args = merge_args(BASE_ARGS, extra_args)
    rows = []
    for table_type in TABLE_TYPES:
        trials = []
        print('  table=', table_type)
        for trial in range(REPEAT):
            result = run_checked([str(BIN), *args, '--table-type', table_type], ROOT, quiet=True, timeout=180)
            df = parse_result(result.stdout, table_type)
            if df.empty:
                raise RuntimeError(f'No parsed rows for {table_type}')
            df['tx_type'] = df['tx_type'].replace(TX_MAP)
            total = df.groupby('repair_type', as_index=False)['duration_ms'].sum()
            total['table_type'] = table_type
            total['trial'] = trial
            trials.append(total)
        df_all = trim_trial_runs(pd.concat(trials, ignore_index=True))
        df_avg = df_all.groupby(['table_type', 'repair_type'], as_index=False)['duration_ms'].mean()
        rows.append(collapse_repairs(df_avg, table_type))
    out = pd.concat(rows, ignore_index=True)
    out['total_ms'] = out['duration_ms'] / TXN_NUM
    return out


def sweep_factor(csv_stem, x_col, values, make_args_fn):
    csv_path = DATA_DIR / f'{csv_stem}_{RUN_TAG}.csv'
    if CONFIG['force_rerun'] and csv_path.exists():
        csv_path.unlink()
    if csv_path.exists():
        print('Using cached CSV:', csv_path.name)
        return pd.read_csv(csv_path, keep_default_na=False)
    all_rows = []
    for value in values:
        print(f'Running {x_col}={value}')
        df = run_single(make_args_fn(value))
        df[x_col] = value
        all_rows.append(df)
    out = pd.concat(all_rows, ignore_index=True)
    out.to_csv(csv_path, index=False)
    print('Saved', csv_path.name)
    return out


def delta_args(delta_ratio):
    analytical = SWEEP['analytical_ratio']
    update = 1.0 - analytical
    remaining = analytical - delta_ratio
    return [
        '--txn-update-ratio', str(update),
        '--txn-delta-ratio', str(delta_ratio),
        '--txn-probe-ratio', str(remaining / 2.0),
        '--txn-scan-ratio', str(remaining / 2.0),
        '--txn-gc-ratio', str(CONFIG['txn_gc_ratio']),
    ]


def update_args(update_ratio):
    return [
        '--analytical-ratio', str(SWEEP['analytical_ratio']),
        '--txn-gc-ratio', str(CONFIG['txn_gc_ratio']),
        '--update-ratio', str(update_ratio),
    ]


def reuse_args(reuse_ratio):
    return [
        '--analytical-ratio', str(SWEEP['analytical_ratio']),
        '--txn-gc-ratio', str(CONFIG['txn_gc_ratio']),
        '--scan-reuse-ratio', str(reuse_ratio),
    ]


df_delta = sweep_factor('sigmod_exp2_delta', 'delta_ratio', SWEEP['delta_values'], delta_args)
df_update = sweep_factor('sigmod_exp2_update', 'update_ratio', SWEEP['update_values'], update_args)
df_reuse = sweep_factor('sigmod_exp2_reuse', 'reuse_ratio', SWEEP['reuse_values'], reuse_args)

display(df_delta.head())


In [ ]:
def plot_one(ax, df, x_col, xlabel, title):
    for key in PLOT_SERIES:
        label, color, linestyle, marker = STYLE[key]
        table_type, repair_type = key
        sub = df[(df['table_type'] == table_type) & (df['repair_type'] == repair_type)].sort_values(x_col)
        if sub.empty:
            continue
        ax.plot(sub[x_col], sub['total_ms'], color=color, linestyle=linestyle, marker=marker, linewidth=1.8, markersize=5, label=label)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Duration (ms / tx)')
    ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)

fig, axes = plt.subplots(1, 3, figsize=(13.6, 4.2), sharey=False)
plot_one(axes[0], df_delta, 'delta_ratio', 'Delta-Scan Share', 'Delta Sweep')
plot_one(axes[1], df_update, 'update_ratio', 'Update Intensity', 'Update Sweep')
plot_one(axes[2], df_reuse, 'reuse_ratio', 'History-Scan Reuse', 'Reuse Sweep')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=5, bbox_to_anchor=(0.5, 1.08), framealpha=0.95)
fig.tight_layout()
out_pdf = FIGS_DIR / f'sigmod_exp2_crossover_{RUN_TAG}.pdf'
fig.savefig(out_pdf, format='pdf')
plt.show()
print('Saved', out_pdf)
